# Datenaufbereitung: CO₂-Emissionen in Deutschland (1990–2024)

In diesem Notebook werden die Rohdaten des Umweltbundesamtes (UBA)
aufbereitet und in eine analysefähige Struktur überführt.

Ziel ist es, aus den komplexen Excel-Rohdaten einen sauberen,
reproduzierbaren Datensatz im Long-Format zu erzeugen, der als
Grundlage für die Analyse und Visualisierung dient.

In [ ]:
import pandas as pd
import re

In [23]:
pfad = "C:/Users/leofi/Desktop/IBM_Data_Analyst/GitHub/co2-analyse-deutschland/daten/original/Emissionsübersichten_KSG-Sektoren_1990–2024.xlsx"

In [24]:
xls = pd.ExcelFile(pfad)
xls.sheet_names

['THG-Trends',
 'THG-Anteile',
 'THG kurz',
 'THG',
 'CO2',
 'CH4',
 'N2O',
 'F-Gase',
 'Daten Sektorgrafik',
 'Sektorgrafik UBA_CI',
 'Daten Zielpfadgrafik',
 'Grafik Zielpfad',
 'Grafik Zielpfadänderung JEGM',
 'Daten Sektor Energiew.',
 'Grafik Sektor Energiew.',
 'Daten Sektor Industrie',
 'Grafik Sektor Industrie',
 'Daten Sektor Gebäude',
 'Grafik Sektor Gebäude',
 'Daten Sektor Verkehr',
 'Grafik Sektor Verkehr',
 'Daten Sektor Landwirtschaft',
 'Grafik Sektor Landwirtschaft',
 'Daten Sektor Abfallwirtschaft',
 'Grafik Sektor Abfallwirtschaft',
 'Unsicherheiten']

## 1. Laden der Rohdaten

Die Rohdaten stammen aus einer Excel-Datei des Umweltbundesamtes.
Die Datei enthält mehrere Tabellenblätter sowie zusätzliche
Meta-Informationen und ist nicht direkt analysefähig.

In [25]:
df_raw = pd.read_excel(
    pfad,
    sheet_name="THG",
    header=None
)

df_raw.head(10)

,0,1,2,3,4,5,6,7,8,9,...,28,29,30,31,32,33,34,35,36,37
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Treibhausgas-Emissionen [tausend Tonnen CO2-äq...,kt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Sektor des Klimaschutzgesetzes (KSG),NaN,1990-01-01 00:00:00,1991-01-01 00:00:00,1992-01-01 00:00:00,1993-01-01 00:00:00,1994-01-01 00:00:00,1995-01-01 00:00:00,1996-01-01 00:00:00,...,2015-01-01 00:00:00,2016-01-01 00:00:00,2017-01-01 00:00:00,2018-01-01 00:00:00,2019-01-01 00:00:00,2020-01-01 00:00:00,2021-01-01 00:00:00,2022-01-01 00:00:00,2023-01-01 00:00:00,2024-01-01 00:00:00
4,NaN,Gesamtemissionen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,ohne LULUCF,Summe,1252397.337977,1206567.46245,1157074.412881,1147917.374271,1129646.310333,1122690.580797,1140026.496404,...,901825.998189,897287.226604,882232.053673,852858.341892,798048.051347,732993.341731,761426.754981,748792.713414,672020.324528,649058.659292
6,NaN,mit LULUCF,Summe,1288424.636136,1193733.176169,1139554.499819,1122594.756579,1112282.931274,1116100.533668,1132834.926421,...,907557.434689,906163.773949,886219.010911,936055.378519,870169.076385,809649.154295,824513.985627,824396.179467,740673.128426,700347.356688
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,1 - Energiewirtschaft,Summe,474772.204321,459944.857888,435675.008357,425924.888382,420014.321917,406935.982252,412872.037054,...,351321.400855,346328.717614,326500.691044,310851.066462,258283.02964,219037.654281,246421.430454,256670.41826,202582.417052,184993.886343
9,NaN,CRF 1.A.1 - Energiewirtschaft,Summe,431082.90739,417239.366857,395545.593952,384453.073047,381961.683212,370192.090228,377049.692117,...,340419.7934,336644.912389,317009.31662,302856.737568,252369.915676,214168.056752,241577.528914,251566.389246,198061.644293,180581.473393


## 2. Erste Datenbereinigung

Die Excel-Tabelle enthält:
- mehrzeilige Überschriften
- Meta- und Summenspalten
- Trennzeilen ohne inhaltliche Bedeutung

In diesem Schritt werden:
- die relevante Header-Zeile gesetzt
- nicht benötigte Meta- und Summenspalten entfernt
- leere Trennzeilen gelöscht

In [26]:
df_raw.columns = df_raw.iloc[3]

df = df_raw.iloc[5:].reset_index(drop=True)

df.head()

3,NaN,Sektor des Klimaschutzgesetzes (KSG),NaN,1990-01-01 00:00:00,1991-01-01 00:00:00,1992-01-01 00:00:00,1993-01-01 00:00:00,1994-01-01 00:00:00,1995-01-01 00:00:00,1996-01-01 00:00:00,...,2015-01-01 00:00:00,2016-01-01 00:00:00,2017-01-01 00:00:00,2018-01-01 00:00:00,2019-01-01 00:00:00,2020-01-01 00:00:00,2021-01-01 00:00:00,2022-01-01 00:00:00,2023-01-01 00:00:00,2024-01-01 00:00:00
0,NaN,ohne LULUCF,Summe,1252397.337977,1206567.46245,1157074.412881,1147917.374271,1129646.310333,1122690.580797,1140026.496404,...,901825.998189,897287.226604,882232.053673,852858.341892,798048.051347,732993.341731,761426.754981,748792.713414,672020.324528,649058.659292
1,NaN,mit LULUCF,Summe,1288424.636136,1193733.176169,1139554.499819,1122594.756579,1112282.931274,1116100.533668,1132834.926421,...,907557.434689,906163.773949,886219.010911,936055.378519,870169.076385,809649.154295,824513.985627,824396.179467,740673.128426,700347.356688
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,1 - Energiewirtschaft,Summe,474772.204321,459944.857888,435675.008357,425924.888382,420014.321917,406935.982252,412872.037054,...,351321.400855,346328.717614,326500.691044,310851.066462,258283.02964,219037.654281,246421.430454,256670.41826,202582.417052,184993.886343
4,NaN,CRF 1.A.1 - Energiewirtschaft,Summe,431082.90739,417239.366857,395545.593952,384453.073047,381961.683212,370192.090228,377049.692117,...,340419.7934,336644.912389,317009.31662,302856.737568,252369.915676,214168.056752,241577.528914,251566.389246,198061.644293,180581.473393


In [28]:
df.columns

Index([                                   nan,
       'Sektor des Klimaschutzgesetzes (KSG)',
                                          nan,
                          1990-01-01 00:00:00,
                          1991-01-01 00:00:00,
                          1992-01-01 00:00:00,
                          1993-01-01 00:00:00,
                          1994-01-01 00:00:00,
                          1995-01-01 00:00:00,
                          1996-01-01 00:00:00,
                          1997-01-01 00:00:00,
                          1998-01-01 00:00:00,
                          1999-01-01 00:00:00,
                          2000-01-01 00:00:00,
                          2001-01-01 00:00:00,
                          2002-01-01 00:00:00,
                          2003-01-01 00:00:00,
                          2004-01-01 00:00:00,
                          2005-01-01 00:00:00,
                          2006-01-01 00:00:00,
                          2007-01-01 00:00:00,
             

In [29]:
df = df.drop(columns=[df.columns[0], df.columns[2]])

df = df.rename(columns={"Sektor des Klimaschutzgesetzes (KSG)": "Sektor"})

df = df.dropna(subset=["Sektor"]).reset_index(drop=True)

df.head()

3,Sektor,1990-01-01 00:00:00,1991-01-01 00:00:00,1992-01-01 00:00:00,1993-01-01 00:00:00,1994-01-01 00:00:00,1995-01-01 00:00:00,1996-01-01 00:00:00,1997-01-01 00:00:00,1998-01-01 00:00:00,...,2015-01-01 00:00:00,2016-01-01 00:00:00,2017-01-01 00:00:00,2018-01-01 00:00:00,2019-01-01 00:00:00,2020-01-01 00:00:00,2021-01-01 00:00:00,2022-01-01 00:00:00,2023-01-01 00:00:00,2024-01-01 00:00:00
0,ohne LULUCF,1252397.337977,1206567.46245,1157074.412881,1147917.374271,1129646.310333,1122690.580797,1140026.496404,1104232.831664,1079170.86898,...,901825.998189,897287.226604,882232.053673,852858.341892,798048.051347,732993.341731,761426.754981,748792.713414,672020.324528,649058.659292
1,mit LULUCF,1288424.636136,1193733.176169,1139554.499819,1122594.756579,1112282.931274,1116100.533668,1132834.926421,1095821.207856,1064363.794624,...,907557.434689,906163.773949,886219.010911,936055.378519,870169.076385,809649.154295,824513.985627,824396.179467,740673.128426,700347.356688
2,1 - Energiewirtschaft,474772.204321,459944.857888,435675.008357,425924.888382,420014.321917,406935.982252,412872.037054,391009.00721,390959.468948,...,351321.400855,346328.717614,326500.691044,310851.066462,258283.02964,219037.654281,246421.430454,256670.41826,202582.417052,184993.886343
3,CRF 1.A.1 - Energiewirtschaft,431082.90739,417239.366857,395545.593952,384453.073047,381961.683212,370192.090228,377049.692117,355969.138291,358861.676722,...,340419.7934,336644.912389,317009.31662,302856.737568,252369.915676,214168.056752,241577.528914,251566.389246,198061.644293,180581.473393
4,CRF 1.A.3.e - Erdgasverdichter,1102.104057,1158.822434,1146.256222,1212.321412,1234.066403,1346.707052,1506.574449,1438.946381,1450.105168,...,1247.283071,1060.152446,1268.246213,1346.983101,1209.89869,777.68308,847.240223,1345.385656,948.32683,875.693769


In [30]:
df = df.rename(columns={col: col.year for col in df.columns if col != "Sektor"})
df.head()

3,Sektor,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,ohne LULUCF,1252397.337977,1206567.46245,1157074.412881,1147917.374271,1129646.310333,1122690.580797,1140026.496404,1104232.831664,1079170.86898,...,901825.998189,897287.226604,882232.053673,852858.341892,798048.051347,732993.341731,761426.754981,748792.713414,672020.324528,649058.659292
1,mit LULUCF,1288424.636136,1193733.176169,1139554.499819,1122594.756579,1112282.931274,1116100.533668,1132834.926421,1095821.207856,1064363.794624,...,907557.434689,906163.773949,886219.010911,936055.378519,870169.076385,809649.154295,824513.985627,824396.179467,740673.128426,700347.356688
2,1 - Energiewirtschaft,474772.204321,459944.857888,435675.008357,425924.888382,420014.321917,406935.982252,412872.037054,391009.00721,390959.468948,...,351321.400855,346328.717614,326500.691044,310851.066462,258283.02964,219037.654281,246421.430454,256670.41826,202582.417052,184993.886343
3,CRF 1.A.1 - Energiewirtschaft,431082.90739,417239.366857,395545.593952,384453.073047,381961.683212,370192.090228,377049.692117,355969.138291,358861.676722,...,340419.7934,336644.912389,317009.31662,302856.737568,252369.915676,214168.056752,241577.528914,251566.389246,198061.644293,180581.473393
4,CRF 1.A.3.e - Erdgasverdichter,1102.104057,1158.822434,1146.256222,1212.321412,1234.066403,1346.707052,1506.574449,1438.946381,1450.105168,...,1247.283071,1060.152446,1268.246213,1346.983101,1209.89869,777.68308,847.240223,1345.385656,948.32683,875.693769


## 3. Umwandlung ins Long-Format

Die Jahreswerte liegen zunächst im sogenannten Wide-Format vor
(jedes Jahr als eigene Spalte).

Für Analyse und Visualisierung werden die Daten in ein Long-Format
überführt, bei dem jede Zeile eine Beobachtung
(Sektor – Jahr – Emissionswert) darstellt.

In [33]:
df_long = df.melt(
    id_vars="Sektor",
    var_name="Jahr",
    value_name="CO2_Emissionen_Mio_t"
)

df_long.head(20)

,Sektor,Jahr,CO2_Emissionen_Mio_t
0,ohne LULUCF,1990,1252397.337977
1,mit LULUCF,1990,1288424.636136
2,1 - Energiewirtschaft,1990,474772.204321
3,CRF 1.A.1 - Energiewirtschaft,1990,431082.90739
4,CRF 1.A.3.e - Erdgasverdichter,1990,1102.104057
5,CRF 1.B - Diffuse Emissionen aus Brennstoffen,1990,42587.192874
6,2 - Industrie,1990,277703.082814
7,CRF 1.A.2 - Verarbeitendes Gewerbe,1990,184425.160743
8,CRF 2.A - Herstellung mineralischer Produkte,1990,23522.377003
9,CRF 2.B - Chemische Industrie,1990,32360.362658


## 4. Auswahl relevanter Sektoren

Der Datensatz enthält mehrere Aggregationsebenen, darunter:
- nationale Gesamtwerte (mit und ohne LULUCF)
- Hauptsektoren gemäß Bundes-Klimaschutzgesetz (KSG)
- detaillierte Unterkategorien (CRF-Codes)

Für die weitere Analyse werden ausschließlich:
- nationale Gesamtwerte
- aggregierte Hauptsektoren (KSG)

berücksichtigt. Detailkategorien werden ausgeschlossen,
um die Vergleichbarkeit und Übersichtlichkeit der Analyse zu gewährleisten.

In [ ]:
df_long_clean = df_long[
    df_long["Sektor"].str.match(r"^(mit LULUCF|ohne LULUCF|\d+\s-)")
].copy()

df_long_clean.head(20)

,Sektor,Jahr,CO2_Emissionen_Mio_t
0,ohne LULUCF,1990,1252397.337977
1,mit LULUCF,1990,1288424.636136
2,1 - Energiewirtschaft,1990,474772.204321
6,2 - Industrie,1990,277703.082814
12,3 - Gebäude,1990,210027.31691
16,4 - Verkehr,1990,163355.366566
21,5 - Landwirtschaft,1990,84989.159157
30,6 - Abfallwirtschaft und Sonstiges,1990,41550.20821
35,"7 - Landnutzung, Landnutzungsänderung und Fors...",1990,36027.298159
42,ohne LULUCF,1991,1206567.46245


## 5. Export des aufbereiteten Datensatzes

Der bereinigte Datensatz wird als CSV-Datei gespeichert.
Diese Datei dient als feste, reproduzierbare Grundlage
für die weitere Analyse in einem separaten Notebook.

In [38]:
export_pfad = "C:/Users/leofi/Desktop/IBM_Data_Analyst/GitHub/co2-analyse-deutschland/daten/aufbereitet/co2_emissionen_deutschland.csv"

df_long_clean.to_csv(
    export_pfad,
    sep=";",
    index=False,
    encoding="utf-8"
)

## Ergebnis

Das Ergebnis dieses Notebooks ist ein sauber aufbereiteter,
analysefähiger Datensatz im Long-Format.

Die Trennung von Datenaufbereitung und Analyse erhöht
die Nachvollziehbarkeit und Reproduzierbarkeit des Projekts.
